# Kimi-Linear KDA State Experiment

Test if Kimi-Linear's KDA (linear attention) state can compress context.

**Requirements:**
- 8× L4 GPUs (GCP)
- vLLM with Kimi-Linear support

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q vllm torch transformers

In [ ]:
# Check GPU availability
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f}GB)")

## 2. Load Kimi-Linear

In [ ]:
from vllm import LLM, SamplingParams

# Configuration for 8× L4
MODEL_NAME = "moonshotai/Kimi-Linear-Instruct"
TENSOR_PARALLEL = 8  # Change based on your GPU count
MAX_MODEL_LEN = 32768

print(f"Loading {MODEL_NAME}...")
print(f"Tensor Parallel: {TENSOR_PARALLEL}")

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=TENSOR_PARALLEL,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
)

print("Model loaded!")

## 3. Test Basic Generation

In [ ]:
# Basic generation test
prompt = "Hello, my name is Alice and I live in Tokyo. Nice to meet you!"

sampling_params = SamplingParams(
    max_tokens=50,
    temperature=0.0,  # Deterministic
)

outputs = llm.generate([prompt], sampling_params)

print(f"Prompt: {prompt}")
print(f"\nResponse: {outputs[0].outputs[0].text}")

## 4. Full Context vs Anchor Comparison

In [ ]:
# Define context and anchor
CONTEXT = """Hi, my name is Alice and I work as a software engineer at a startup in Tokyo.
I've been living here for 5 years and I really enjoy the city.
My hobbies include hiking, photography, and cooking Japanese food.
Last weekend I went to Mount Fuji and took some amazing photos."""

# 5-word semantic anchor
ANCHOR = "<alice-software-tokyo-hiking-photography/>"

# Instruction for anchor
INSTRUCTION = """[COMPRESSED CONTEXT: The anchor below summarizes previous conversation.]
Previous: """

QUERY = "\n\nUser: What are your hobbies?\nAssistant:"

print(f"Context length: {len(CONTEXT)} chars")
print(f"Anchor: {ANCHOR}")
print(f"Anchor length: {len(ANCHOR)} chars")
print(f"Compression: {len(CONTEXT) / len(ANCHOR):.1f}x")

In [ ]:
# Test 1: Full context
full_prompt = CONTEXT + QUERY
full_output = llm.generate([full_prompt], sampling_params)[0].outputs[0].text

print("FULL CONTEXT RESPONSE:")
print(full_output)

In [ ]:
# Test 2: Anchor only
anchor_prompt = INSTRUCTION + ANCHOR + QUERY
anchor_output = llm.generate([anchor_prompt], sampling_params)[0].outputs[0].text

print("ANCHOR ONLY RESPONSE:")
print(anchor_output)

In [ ]:
# Compare responses
print("=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"\nFull context: {full_output[:100]}...")
print(f"\nAnchor only:  {anchor_output[:100]}...")

# Check if key info preserved
keywords = ['hiking', 'photography', 'cooking', 'japanese']
print(f"\nKeyword preservation:")
for kw in keywords:
    in_full = kw in full_output.lower()
    in_anchor = kw in anchor_output.lower()
    print(f"  {kw}: Full={in_full}, Anchor={in_anchor}")

## 5. Explore KDA State Access

In [ ]:
# Explore what KDA-related APIs are available
engine = llm.llm_engine

print("Searching for KDA/state-related attributes...\n")

# Check engine
print("Engine attributes with 'kv', 'kda', 'state', 'cache':")
for attr in dir(engine):
    if any(x in attr.lower() for x in ['kv', 'kda', 'state', 'cache']):
        print(f"  - engine.{attr}")

In [ ]:
# Try to access model internals
try:
    model = engine.model_executor.driver_worker.model_runner.model
    print(f"Model type: {type(model)}")
    print(f"\nModel attributes with 'kda', 'state', 'linear':")
    for attr in dir(model):
        if any(x in attr.lower() for x in ['kda', 'state', 'linear', 'recurrent']):
            print(f"  - model.{attr}")
except Exception as e:
    print(f"Could not access model: {e}")

## 6. Long Context Test

In [ ]:
# Test with longer context to see KDA benefits
LONG_CONTEXT = CONTEXT * 10  # Repeat 10 times

print(f"Long context length: {len(LONG_CONTEXT)} chars")

# Generate with long context
long_prompt = LONG_CONTEXT + QUERY
long_output = llm.generate([long_prompt], sampling_params)[0].outputs[0].text

print(f"\nLong context response: {long_output}")

In [ ]:
# Compare anchor vs long context
print("=" * 60)
print("LONG CONTEXT vs ANCHOR")
print("=" * 60)
print(f"\nLong context ({len(LONG_CONTEXT)} chars): {long_output[:100]}...")
print(f"\nAnchor ({len(ANCHOR)} chars): {anchor_output[:100]}...")
print(f"\nCompression ratio: {len(LONG_CONTEXT) / len(ANCHOR):.0f}x")

## 7. Summary

In [ ]:
print("""
======================================================================
EXPERIMENT SUMMARY
======================================================================

Key findings:
1. Does anchor preserve semantic meaning? [Check outputs above]
2. Is KDA state directly accessible? [Check exploration above]
3. Does compression work at scale? [Check long context test]

Next steps:
- If KDA state accessible: Implement save/restore
- If not accessible: Modify vLLM source or file feature request
- Test with even longer contexts (100K+ tokens)
""")